# 02 — LLM routing and reliability

**Goal:** separate the decision about *which capability is needed* from the code that calls a provider.

```text
question → classify complexity → route dictionary → model client → provider
```

## The route is configuration, not the model

The router returns instructions such as model name, output budget, and reasoning control. The client translates those instructions into a real model call. This keeps business decisions independent from Gemini, Groq, or a future provider.

In [ ]:
ROUTES = {
    "simple": {"model": "groq/llama-3.1-8b-instant", "max_tokens": 80},
    "general": {
        "model": "gemini/gemini-2.5-flash",
        "max_tokens": 500,
        "thinking": {"type": "disabled", "budget_tokens": 0},
    },
    "complex": {
        "model": "groq/openai/gpt-oss-20b",
        "max_tokens": 1000,
        "reasoning_effort": "medium",
    },
}


def choose_route(complexity: str) -> dict:
    if complexity not in ROUTES:
        raise ValueError(f"Unknown complexity: {complexity}")
    return ROUTES[complexity]


print(choose_route("general"))

## One router, different naming conventions

LiteLLM uses `provider/model`. LangChain's initializer uses `provider:model`. Split only at the first slash because the model name may contain another slash.

In [ ]:
PROVIDER_NAMES = {"gemini": "google_genai"}


def to_langchain_identifier(route: dict) -> str:
    provider, model_name = route["model"].split("/", maxsplit=1)
    provider = PROVIDER_NAMES.get(provider, provider)
    return f"{provider}:{model_name}"


print(to_langchain_identifier(choose_route("general")))
print(to_langchain_identifier(choose_route("complex")))

## Retries and error boundaries

- **Timeout:** stop waiting forever.
- **Retry:** repeat a transient failure a small bounded number of times.
- **Error translation:** convert provider-specific failures into one application error such as `LLMServiceError`.
- Do not retry invalid input or a programming bug automatically.

In [ ]:
class TemporaryProviderError(Exception):
    pass


attempts = 0


def fake_provider_call() -> str:
    global attempts
    attempts += 1
    if attempts < 3:
        raise TemporaryProviderError("try again")
    return "provider answer"


def call_with_bounded_retry(max_attempts: int = 3) -> str:
    for attempt in range(1, max_attempts + 1):
        try:
            return fake_provider_call()
        except TemporaryProviderError:
            if attempt == max_attempts:
                raise


print(call_with_bounded_retry(), "after", attempts, "attempts")

## YOUR TURN

1. Add a `local` route using `ollama/qwen3`.
2. Predict the identifier produced for `groq/openai/gpt-oss-20b` before running it.
3. Explain why the prompt should not contain a hardcoded provider name.